# GPU tests and benchmarks (Colab / Kaggle T4)

The laptop has no CUDA GPU, so GPU numbers come from here. Runtime -> Change runtime type -> **T4 GPU**, then run all cells.
Results land in `bench/results/*_t4.*`; the last cell zips them for download so they can be committed.

Every number in the README's GPU rows comes from these cells; nothing is typed in by hand.

Already have the Phase 4 results? You can skip the two Phase 4 batching cells.

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/GARVIT-bit/llm-inference-engine.git
%cd llm-inference-engine
!git log --oneline -1

In [ ]:
# torch with CUDA is preinstalled on Colab; don't reinstall it
!pip install -q tiktoken safetensors huggingface_hub pyarrow fastapi uvicorn httpx transformers openai pytest

## Phase 4: correctness on GPU
`test_cuda_fp16_matches_fp32` runs only when CUDA is available (skipped on the laptop).

In [ ]:
!python -m pytest -q tests/test_torch_model.py tests/test_paged_cache.py tests/test_sampler.py tests/test_scheduler.py tests/test_server.py

## Phase 4: static vs continuous batching on the T4 (fp16)
Same workload and settings as the laptop run (64 requests, max batch 16), plus a max-batch-64 run.

In [ ]:
!python bench/batching_bench.py --device cuda --out batching_t4
!cat bench/results/batching_t4.md

In [ ]:
!python bench/batching_bench.py --device cuda --max-batch 64 --num-blocks 1536 --out batching_t4_b64
!cat bench/results/batching_t4_b64.md

## Phase 5: Triton fused attention
Correctness first (kernel vs fp32 reference, and the full model with Triton attention), then speed and memory from seq 256 to 4096.

In [ ]:
!python -m pytest -q tests/test_attention_kernel.py

In [ ]:
!python bench/attention_bench.py --out attention_t4
!cat bench/results/attention_t4.md

## Phase 5b: diagnose and tune the Triton kernel (~10-15 min)
Triton matmul vs cuBLAS on this GPU, what the compiler emitted (tensor-core `mma` or scalar FMAs, register spills), and a block-size / warps / stages sweep. Already have the other Phase 5 results? Run only the setup cells, this one, and the download cell.

In [ ]:
!python bench/attention_tune.py --out attention_tune_t4
!cat bench/results/attention_tune_t4.md

## Phase 5: speculative decoding on the T4 (fp16)
Downloads gpt2-medium (~1.5 GB) on first use. The strict test (speculative == target greedy, fp32) runs on the laptop; here the bench reports whether fp16 output still matches target greedy.

In [ ]:
!python bench/speculative_bench.py --device cuda --out speculative_t4
!cat bench/results/speculative_t4.md

## Download results
Unzip into the repo's `bench/results/` and commit.

In [ ]:
!zip -j gpu_results.zip bench/results/*_t4*
from google.colab import files
files.download('gpu_results.zip')